In [ ]:
# in databricks, I need to read csv file and append the record to the existig table name, 
# how can we achive this

# To ingest CSV files from Azure Data Lake Storage Gen2 (ADLS Gen2) into Databricks and 
# append them to an existing table

configs = {
  "fs.azure.account.auth.type": "OAuth",
  "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
  "fs.azure.account.oauth2.client.id": "<client-id>",
  "fs.azure.account.oauth2.client.secret": "<client-secret>",
  "fs.azure.account.oauth2.client.endpoint": "https://login.microsoftonline.com/<tenant-id>/oauth2/token"
}

dbutils.fs.mount(
  source = "abfss://<container>@<storage-account>.dfs.core.windows.net/",
  mount_point = "/mnt/adls",
  extra_configs = configs)



# Read CSV file into DataFrame
df = spark.read.format("csv") \
    .option("header", "true") \   # if CSV has headers
    .option("inferSchema", "true") \
    .load("/mnt/data/new_records.csv")

# Append to existing Delta table
df.write.format("delta") 
    .option("header", "true")
    .option("mergeSchema", "true") 
    .mode("append") \
    .saveAsTable("existing_table_name")


# working with ETL pipelines and Delta tables, you can also use Auto Loader for continuous 
# ingestion if new CSV files arrive regularly:
df = (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("header", "true")
      .load("/mnt/data/csv_folder"))

df.writeStream.format("delta") \
    .option("checkpointLocation", "/mnt/checkpoints/csv_ingest") \
    .outputMode("append") \
    .table("existing_table_name")


PySpark program Calculate total sales amount per product (quantity × price).
    Find the top 2 products with the highest sales amount.
    data = [
    (1, 101, 'Laptop', 2, 1000.0),
    (2, 102, 'Mouse', 5, 25.0),
    (3, 101, 'Laptop', 1, 1000.0),
    (4, 103, 'Keyboard', 3, 75.0),
    (5, 102, 'Mouse', 2, 25.0),
]

columns = ['order_id', 'product_id', 'product_name', 'quantity', 'price']

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Initialize Spark
spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()

# Sample data
data = [
    (1, 101, 'Laptop', 2, 1000.0),
    (2, 102, 'Mouse', 5, 25.0),
    (3, 101, 'Laptop', 1, 1000.0),
    (4, 103, 'Keyboard', 3, 75.0),
    (5, 102, 'Mouse', 2, 25.0),
]

columns = ['order_id', 'product_id', 'product_name', 'quantity', 'price']

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Calculate sales amount per row
df = df.withColumn("sales_amount", F.col("quantity") * F.col("price"))

# Aggregate total sales per product
df_total = df.groupBy("product_id", "product_name") \
             .agg(F.sum("sales_amount").alias("total_sales"))

# Find top 2 products by sales amount
top_products = df_total.orderBy(F.desc("total_sales")).limit(2)

top_products.show()
